In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.datasets as datasets
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, random_split, Subset
import time
import matplotlib.pyplot as plt
import os
from sklearn.model_selection import train_test_split


import torch
import pyro
import pyro.distributions as dist
from pyro.nn.module import PyroModule, PyroParam
from pyro.infer.autoguide import AutoGuide
from pyro.infer.autoguide.initialization import InitMessenger, init_to_feasible
from pyro.distributions import constraints
from contextlib import ExitStack

import pickle
from tqdm import tqdm
import copy

import pyro
import pyro.distributions as dist
from pyro.nn import PyroModule, PyroSample
from pyro.infer.autoguide import AutoNormal

import pandas as pd

import numpy as np
from sklearn.metrics import confusion_matrix

from bitflip import bitflip_float32

from torchvision.datasets import ImageFolder

import os

import json

c:\Users\Revalda Putawara\.conda\envs\bnntest\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [22]:
shipsnet_mean = [0.4119, 0.4243, 0.3724]
shipsnet_std = [0.1899, 0.1569, 0.1515]

def load_data(batch_size=16):
    transform = transforms.Compose([
        transforms.Resize((64, 64)),
        transforms.ToTensor(),
        transforms.Normalize(mean=shipsnet_mean, 
                             std=shipsnet_std)
    ])

    #dataset = datasets.EuroSAT(root='./data', transform=transform, download=True)
    dataset = ImageFolder(
    root="data/shipsnet/foldered",
    transform=transform
    )
    torch.manual_seed(42)

    #train_size = int(0.8 * len(dataset))
    #test_size = len(dataset) - train_size
    #train_dataset, test_dataset = random_split(dataset, [train_size, test_size])
    
    with open('datasplit/shipsnet_split_indices.pkl', 'rb') as f:
        split = pickle.load(f)
        train_dataset = Subset(dataset, split['train'])
        test_dataset = Subset(dataset, split['test'])

    # Add num_workers and pin_memory for faster data loading
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, 
                             num_workers=4, pin_memory=True, persistent_workers=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size,
                            num_workers=4, pin_memory=True, persistent_workers=True)
    return train_loader, test_loader

from pyro.distributions.util       import sum_rightmost
from pyro.ops.tensor_utils         import periodic_repeat
from pyro.distributions.transforms import biject_to
from pyro.infer.autoguide.utils    import (
    deep_setattr,
    deep_getattr,
    helpful_support_errors,
)

import pyro.poutine as poutine

from pyro.infer.autoguide import AutoGuideList  #, AutoLowRankMultivariateNormal\
from pyro.infer.autoguide import AutoLowRankMultivariateNormal


class AutoLaplace(AutoGuide):
    """
    An AutoGuide that uses a Laplace(loc, scale) marginal for each latent.
    """
    scale_constraint = constraints.softplus_positive

    def __init__(
        self, model, *, init_loc_fn=init_to_feasible, init_scale=0.1, create_plates=None
    ):
        self.init_loc_fn = init_loc_fn
        if not isinstance(init_scale, float) or not (init_scale > 0):
            raise ValueError(f"Expected init_scale > 0, got {init_scale}")
        self._init_scale = init_scale

        model = InitMessenger(self.init_loc_fn)(model)
        super().__init__(model, create_plates=create_plates)

    def _setup_prototype(self, *args, **kwargs):
        super()._setup_prototype(*args, **kwargs)

        self._event_dims = {}
        self.locs = PyroModule()
        self.scales = PyroModule()

        for name, site in self.prototype_trace.iter_stochastic_nodes():
            # ← use helpful_support_errors directly
            with helpful_support_errors(site):
                init_loc = (
                    biject_to(site["fn"].support)
                    .inv(site["value"].detach())
                    .detach()
                )
            event_dim = site["fn"].event_dim + init_loc.dim() - site["value"].dim()
            self._event_dims[name] = event_dim

            # handle subsampling plates
            for frame in site["cond_indep_stack"]:
                full_size = frame.full_size or frame.size
                if full_size != frame.size:
                    dim = frame.dim - event_dim
                    init_loc = periodic_repeat(init_loc, full_size, dim).contiguous()

            init_scale = torch.full_like(init_loc, self._init_scale)

            deep_setattr(
                self.locs, name, PyroParam(init_loc, constraints.real, event_dim)
            )
            deep_setattr(
                self.scales,
                name,
                PyroParam(init_scale, self.scale_constraint, event_dim),
            )

    def _get_loc_and_scale(self, name):
        site_loc = deep_getattr(self.locs, name)
        site_scale = deep_getattr(self.scales, name)
        return site_loc, site_scale

    def forward(self, *args, **kwargs):
        if self.prototype_trace is None:
            self._setup_prototype(*args, **kwargs)

        plates = self._create_plates(*args, **kwargs)
        result = {}

        for name, site in self.prototype_trace.iter_stochastic_nodes():
            transform = biject_to(site["fn"].support)

            with ExitStack() as stack:
                for frame in site["cond_indep_stack"]:
                    if frame.vectorized:
                        stack.enter_context(plates[frame.name])

                site_loc, site_scale = self._get_loc_and_scale(name)

                unconstrained = pyro.sample(
                    f"{name}_unconstrained",
                    dist.Laplace(site_loc, site_scale)
                        .to_event(self._event_dims[name]),
                    infer={"is_auxiliary": True},
                )

                value = transform(unconstrained)
                if poutine.get_mask() is False:
                    log_density = 0.0
                else:
                    log_density = transform.inv.log_abs_det_jacobian(
                        value, unconstrained
                    )
                    log_density = sum_rightmost(
                        log_density,
                        log_density.dim() - value.dim() + site["fn"].event_dim,
                    )
                delta = dist.Delta(
                    value,
                    log_density=log_density,
                    event_dim=site["fn"].event_dim,
                )
                result[name] = pyro.sample(name, delta)

        return result

    @torch.no_grad()
    def median(self, *args, **kwargs):
        medians = {}
        for name, site in self.prototype_trace.iter_stochastic_nodes():
            site_loc, _ = self._get_loc_and_scale(name)
            med = biject_to(site["fn"].support)(site_loc)
            medians[name] = med.clone() if med is site_loc else med
        return medians

    @torch.no_grad()
    def quantiles(self, quantiles, *args, **kwargs):
        results = {}
        for name, site in self.prototype_trace.iter_stochastic_nodes():
            site_loc, site_scale = self._get_loc_and_scale(name)
            qs = torch.tensor(quantiles, dtype=site_loc.dtype, device=site_loc.device)
            qs = qs.reshape((-1,) + (1,) * site_loc.dim())
            qvals = dist.Laplace(site_loc, site_scale).icdf(qs)
            results[name] = biject_to(site["fn"].support)(qvals)
        return results


class UniformReal(dist.Uniform):
    @property
    def support(self):
        return constraints.real

class AutoUniform(AutoGuide):
    """
    An AutoGuide that uses a Uniform(low, low+width) marginal for each latent.
    """
    # `width` must be positive
    width_constraint = constraints.softplus_positive

    def __init__(
        self, model, *, init_loc_fn=init_to_feasible, init_scale=0.1, create_plates=None
    ):
        self.init_loc_fn = init_loc_fn
        if not isinstance(init_scale, float) or not (init_scale > 0):
            raise ValueError(f"Expected init_scale > 0, got {init_scale}")
        self._init_scale = init_scale

        model = InitMessenger(self.init_loc_fn)(model)
        super().__init__(model, create_plates=create_plates)

    def _setup_prototype(self, *args, **kwargs):
        super()._setup_prototype(*args, **kwargs)
        self._event_dims = {}
        self.lows = PyroModule()
        self.widths = PyroModule()

        for name, site in self.prototype_trace.iter_stochastic_nodes():
            # 1. get an unconstrained init_loc (inverse‐transform of site["value"])
            with helpful_support_errors(site):
                init_loc = (
                    biject_to(site["fn"].support)
                    .inv(site["value"].detach())
                    .detach()
                )
            event_dim = site["fn"].event_dim + init_loc.dim() - site["value"].dim()
            self._event_dims[name] = event_dim

            # 2. if subsampled, expand back to full size
            for frame in site["cond_indep_stack"]:
                full_size = frame.full_size or frame.size
                if full_size != frame.size:
                    dim = frame.dim - event_dim
                    init_loc = periodic_repeat(init_loc, full_size, dim).contiguous()

            # 3. build initial low & width around that init_loc
            init_low   = init_loc - self._init_scale
            init_width = torch.full_like(init_loc, 2.0 * self._init_scale)

            # 4. register as PyroParams
            deep_setattr(
                self.lows,  name,
                PyroParam(init_low,   constraints.real,             event_dim),
            )
            deep_setattr(
                self.widths, name,
                PyroParam(init_width, self.width_constraint,       event_dim),
            )

    def _get_low_and_width(self, name):
        low   = deep_getattr(self.lows,  name)
        width = deep_getattr(self.widths, name)
        return low, width

    def forward(self, *args, **kwargs):
        if self.prototype_trace is None:
            self._setup_prototype(*args, **kwargs)

        plates = self._create_plates(*args, **kwargs)
        result = {}

        for name, site in self.prototype_trace.iter_stochastic_nodes():
            transform = biject_to(site["fn"].support)
            with ExitStack() as stack:
                for frame in site["cond_indep_stack"]:
                    if frame.vectorized:
                        stack.enter_context(plates[frame.name])

                low, width = self._get_low_and_width(name)
                # draw unconstrained latent from Uniform(low, low + width)
                unconstrained = pyro.sample(
                    f"{name}_unconstrained",
                    #dist.Uniform(low, low + width).to_event(self._event_dims[name]),
                    UniformReal(low, low + width).to_event(self._event_dims[name]),
                    infer={"is_auxiliary": True},
                )

                # map into constrained space
                value = transform(unconstrained)
                if poutine.get_mask() is False:
                    log_density = 0.0
                else:
                    log_density = transform.inv.log_abs_det_jacobian(
                        value, unconstrained
                    )
                    log_density = sum_rightmost(
                        log_density,
                        log_density.dim() - value.dim() + site["fn"].event_dim,
                    )
                delta = dist.Delta(
                    value,
                    log_density=log_density,
                    event_dim=site["fn"].event_dim,
                )
                result[name] = pyro.sample(name, delta)

        return result

    @torch.no_grad()
    def median(self, *args, **kwargs):
        """
        Posterior median is just the 0.5‐quantile of Uniform = low + 0.5*width
        """
        medians = {}
        for name, site in self.prototype_trace.iter_stochastic_nodes():
            low, width = self._get_low_and_width(name)
            med = biject_to(site["fn"].support)(low + 0.5 * width)
            medians[name] = med.clone() if med is low else med
        return medians

    @torch.no_grad()
    def quantiles(self, quantiles, *args, **kwargs):
        """
        Posterior quantiles via Uniform.icdf(q).
        """
        results = {}
        qs = torch.tensor(quantiles)
        for name, site in self.prototype_trace.iter_stochastic_nodes():
            low, width = self._get_low_and_width(name)
            # shape: [len(quantiles), *low.shape]
            qvals = UniformReal(low, low + width).icdf(qs.reshape((-1,) + (1,) * low.dim()))
            #qvals = dist.Uniform(low, low + width).icdf(qs.reshape((-1,) + (1,) * low.dim()))
            results[name] = biject_to(site["fn"].support)(qvals)
        return results


import math
import torch.nn as nn
import pyro.distributions as dist
from pyro.nn import PyroParam
from pyro.distributions import constraints
from pyro.infer.autoguide import AutoContinuous
from pyro.infer.autoguide.initialization import init_to_median

class AutoLowRankMultivariateLaplace(AutoContinuous):
    """
    Low-rank-plus-diagonal multivariate Laplace guide.
    Usage::
        guide = AutoLowRankLaplace(model, rank=10)
        svi = SVI(model, guide, ...)
    """
    scale_constraint = constraints.softplus_positive

    def __init__(self, model, init_loc_fn=init_to_median, init_scale=0.1, rank=None):
        if not isinstance(init_scale, float) or not (init_scale > 0):
            raise ValueError(f"Expected init_scale > 0 but got {init_scale}")
        if not (rank is None or isinstance(rank, int) and rank > 0):
            raise ValueError(f"Expected rank > 0 but got {rank}")
        self._init_scale = init_scale
        self.rank = rank
        super().__init__(model, init_loc_fn=init_loc_fn)

    def _setup_prototype(self, *args, **kwargs):
        super()._setup_prototype(*args, **kwargs)
        # location
        self.loc = nn.Parameter(self._init_loc())
        # default rank ≈ sqrt(latent_dim)
        if self.rank is None:
            self.rank = int(round(self.latent_dim**0.5))
        # base (diagonal) scale
        self.scale_base = PyroParam(
            self.loc.new_full((self.latent_dim,), 0.5**0.5 * self._init_scale),
            constraint=self.scale_constraint
        )
        # low-rank factor
        self.cov_factor = nn.Parameter(
            self.loc.new_empty(self.latent_dim, self.rank)
                .normal_(0, 1 / math.sqrt(self.rank))
        )

    def get_posterior(self, *args, **kwargs):
        """
        Returns a multivariate Laplace with ‘effective’ scale
        incorporating low-rank + diagonal structure.
        """
        base = self.scale_base
        # apply scale to factor → [latent_dim x rank]
        factor = self.cov_factor * base.unsqueeze(-1)
        # effective per-dimension scale: base * sqrt(1 + sum_j factor^2_ij)
        eff_scale = base * (factor.pow(2).sum(-1) + 1).sqrt()
        return dist.Laplace(self.loc, eff_scale)

    def _loc_scale(self, *args, **kwargs):
        base = self.scale_base
        factor = self.cov_factor * base.unsqueeze(-1)
        eff_scale = base * (factor.pow(2).sum(-1) + 1).sqrt()
        return self.loc, eff_scale


class AutoLowRankMultivariateUniform(AutoContinuous):
    """
    Low-rank-plus-diagonal multivariate Uniform guide,
    parameterized by center (loc) ± half‐range.
    Usage::
        guide = AutoLowRankUniform(model, rank=10)
        svi = SVI(model, guide, ...)
    """
    range_constraint = constraints.positive

    def __init__(self, model, init_loc_fn=init_to_median, init_range=0.1, rank=None):
        if not isinstance(init_range, float) or not (init_range > 0):
            raise ValueError(f"Expected init_range > 0 but got {init_range}")
        if not (rank is None or isinstance(rank, int) and rank > 0):
            raise ValueError(f"Expected rank > 0 but got {rank}")
        self._init_range = init_range
        self.rank = rank
        super().__init__(model, init_loc_fn=init_loc_fn)

    def _setup_prototype(self, *args, **kwargs):
        super()._setup_prototype(*args, **kwargs)
        # center
        self.loc = nn.Parameter(self._init_loc())
        if self.rank is None:
            self.rank = int(round(self.latent_dim**0.5))
        # base half‐range (diagonal)
        self.range_base = PyroParam(
            self.loc.new_full((self.latent_dim,), self._init_range),
            constraint=self.range_constraint
        )
        # low-rank factor
        self.cov_factor = nn.Parameter(
            self.loc.new_empty(self.latent_dim, self.rank)
                .normal_(0, 1 / math.sqrt(self.rank))
        )

    def get_posterior(self, *args, **kwargs):
        """
        Returns a multivariate Uniform(loc - Δ, loc + Δ)
        where Δ = base_range * sqrt(1 + sum_j (cov_factor * base_range)^2).
        """
        base = self.range_base
        factor = self.cov_factor * base.unsqueeze(-1)
        # effective half‐range
        half_range = base * (factor.pow(2).sum(-1) + 1).sqrt()
        #return dist.Uniform(self.loc - half_range, self.loc + half_range)
        return UniformReal(self.loc - half_range, self.loc + half_range)

    def _loc_scale(self, *args, **kwargs):
        base = self.range_base
        factor = self.cov_factor * base.unsqueeze(-1)
        half_range = base * (factor.pow(2).sum(-1) + 1).sqrt()
        return self.loc, half_range


class BayesShipsCNN(PyroModule):
    def __init__(
        self,
        num_classes=2,   # now 2 for Categorical
        device=torch.device("cuda"),
        activation='relu',
        prior_dist='gaussian',
        mu=0.0,
        b=1.0,
        prior_params=None
    ):
        super().__init__()
        self.device = device

        # Activation setup
        if isinstance(activation, str):
            act_map = {
                'relu': F.relu,
                'tanh': torch.tanh,
                'sigmoid': torch.sigmoid,
                'sin': torch.sin,
                'relu6': F.relu6,
                'leaky_relu': F.leaky_relu,
                'selu': F.selu,
                'actWG': self.actWG,
                'actRWG': self.actRWG,
            }
            self.activation_fn = act_map[activation]
        elif callable(activation):
            self.activation_fn = activation
        else:
            raise ValueError("activation must be a string or callable")

        # Prior setup
        self.prior_dist = prior_dist
        params = {'mu': mu, 'b': b} if prior_params is None else prior_params
        self.prior_mu = torch.tensor(params['mu'], device=device, dtype=torch.float32)
        self.prior_b  = torch.tensor(params['b'], device=device, dtype=torch.float32)

        print(f"[INFO] Using prior: {self.prior_dist} (mu={self.prior_mu.item()}, b={self.prior_b.item()})")

        # Layers
        self.conv1 = PyroModule[nn.Conv2d](3, 32, kernel_size=3, padding=1)
        self.conv1.weight = PyroSample(self._make_prior([32, 3, 3, 3]))
        self.conv1.bias   = PyroSample(self._make_prior([32]))

        self.conv2 = PyroModule[nn.Conv2d](32, 64, kernel_size=3, padding=1)
        self.conv2.weight = PyroSample(self._make_prior([64, 32, 3, 3]))
        self.conv2.bias   = PyroSample(self._make_prior([64]))

        self.pool = nn.MaxPool2d(2, 2)

        self.fc1 = PyroModule[nn.Linear](64 * 16 * 16, num_classes)
        self.fc1.weight = PyroSample(self._make_prior([num_classes, 64 * 16 * 16]))
        self.fc1.bias   = PyroSample(self._make_prior([num_classes]))

    def actWG(self, x, alpha=1.0):
        return x * torch.exp(-alpha * x ** 2)

    def actRWG(self, x, alpha=1.0):
        wg = x * torch.exp(-alpha * x ** 2)
        return torch.max(torch.zeros_like(wg), wg)

    def _make_prior(self, shape):
        if self.prior_dist == 'gaussian':
            base = dist.Normal(self.prior_mu, self.prior_b)
        elif self.prior_dist == 'laplace':
            base = dist.Laplace(self.prior_mu, self.prior_b)
        elif self.prior_dist == 'uniform':
            #base = dist.Uniform(-self.prior_b, self.prior_b)
            base = UniformReal(-self.prior_b, self.prior_b)
        else:
            raise ValueError(f"Unsupported prior: {self.prior_dist}")
        return base.expand(shape).to_event(len(shape))

    def forward(self, x, y=None):
        x = self.activation_fn(self.conv1(x))
        x = self.pool(x)
        x = self.activation_fn(self.conv2(x))
        x = self.pool(x)

        x = x.view(x.size(0), -1)
        logits = self.fc1(x)  # shape [batch, 2]

        if y is not None:
            with pyro.plate("data", x.size(0)):
                pyro.sample("obs", dist.Categorical(logits=logits), obs=y)
        return logits


def load_model(timestamp):
    config_path = os.path.join(search_dir, config_files[timestamp])
    guide_path = os.path.join(search_dir, guide_files[timestamp])
    model_path = os.path.join(search_dir, model_files[timestamp])
    param_path = os.path.join(search_dir, param_files[timestamp])

    print(f"Loading model with config_path: {config_path}")

    with open(config_path, 'r') as f:
        config = json.load(f)

    model = BayesShipsCNN(
        num_classes=num_classes,
        device=device,
        activation=config['activation'],
        prior_dist=config['prior'],
        mu=config['prior_params']['mu'],
        b=config['prior_params'],
        prior_params=config.get('prior_params', None)
    ).to(device)

    # Load the guide
    #guide = AutoDiagonalNormal(model)

    # Load the model state
    model.load_state_dict(torch.load(model_path))
    
    # Load the guide state
    #guide.load_state_dict(torch.load(guide_path))

    return model, param_path

class NewInjector:
    def __init__(self, trained_model, device, test_loader, num_samples, multivariate_flag=False):
        """
        Initializes SEU injector
        """
        self.device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.trained_model = trained_model.to(self.device)
        self.test_loader = test_loader
        self.trained_model.eval()
        self.num_samples = num_samples

        #self.guide = AutoDiagonalNormal(self.trained_model).to(self.device)
        if self.trained_model.prior_dist == 'gaussian':
            if multivariate_flag:
                self.guide = AutoGuideList(bayesian_model)

                # 1) conv1.weight
                self.guide.add(
                    AutoLowRankMultivariateNormal(
                        poutine.block(bayesian_model, expose=["conv1.weight"]),
                        rank=20,
                        init_scale=0.05,
                        #prefix="AutoGuideList.conv1.weight"
                    )
                )

                # 2) conv1.bias
                self.guide.add(
                    AutoLowRankMultivariateNormal(
                        poutine.block(bayesian_model, expose=["conv1.bias"]),
                        rank=5,
                        init_scale=0.05,
                        #prefix="AutoGuideList.conv1.bias"
                    )
                )

                # 3) conv2.weight
                self.guide.add(
                    AutoLowRankMultivariateNormal(
                        poutine.block(bayesian_model, expose=["conv2.weight"]),
                        rank=20,
                        init_scale=0.05,
                        #prefix="AutoGuideList.conv2.weight"
                    )
                )

                # 4) conv2.bias
                self.guide.add(
                    AutoLowRankMultivariateNormal(
                        poutine.block(bayesian_model, expose=["conv2.bias"]),
                        rank=5,
                        init_scale=0.05,
                        #prefix="AutoGuideList.conv2.bias"
                    )
                )

                # 5) fc1.weight
                self.guide.add(
                    AutoLowRankMultivariateNormal(
                        poutine.block(bayesian_model, expose=["fc1.weight"]),
                        rank=20,
                        init_scale=0.05,
                        #prefix="AutoGuideList.fc1.weight"
                    )
                )

                # 6) fc1.bias
                self.guide.add(
                    AutoLowRankMultivariateNormal(
                        poutine.block(bayesian_model, expose=["fc1.bias"]),
                        rank=5,
                        init_scale=0.05,
                        #prefix="AutoGuideList.fc1.bias"
                    )
                )
            else:
                self.guide = AutoNormal(self.trained_model, init_scale=0.05).to(self.device)
        elif self.trained_model.prior_dist == 'laplace':
            self.guide = AutoLaplace(self.trained_model, init_scale=0.05).to(self.device)
        elif self.trained_model.prior_dist == 'uniform':
            self.guide = AutoUniform(self.trained_model, init_scale=0.05).to(self.device)
        else:
            raise ValueError(f"Unsupported prior: {self.trained_model.prior_dist}")
        
        pyro.get_param_store().clear()
        pyro.get_param_store().set_state(torch.load(pyro_param_store_path, weights_only=False))

        initial_labels, initial_predictions, initial_logits, initial_probs = self.predict_data_probs(self.num_samples)
        self.initial_accuracy = self.return_accuracy(initial_labels, initial_predictions)
        self.initial_probs = np.array(initial_probs)

        print(f"Initial accuracy: {self.initial_accuracy}")

    def predict_data_probs(self, num_samples=10):
        all_labels = []
        all_predictions = []
        all_logits = []
        all_probs = []

        with torch.no_grad():
            for images, labels in tqdm(self.test_loader, desc="Evaluating"):
                images, labels = images.to(self.device), labels.to(self.device)
                logits_mc = torch.zeros(num_samples, images.size(0), self.trained_model.fc1.out_features).to(self.device)

                for i in range(num_samples):
                    guide_trace = pyro.poutine.trace(self.guide).get_trace(images)
                    if i == 0:  # Only print once per batch
                        # Extract 'conv1.weight' sample site from the guide trace
                        if "conv1.bias" in guide_trace.nodes:
                            conv1_bias = guide_trace.nodes["conv1.bias"]["value"]
                            print(f"'conv1.bias' sampled value (batch {images.shape[0]}):")
                            print(conv1_bias)
                        else:
                            print("'conv1.weight' not found in the guide trace.")
                    replayed_model = pyro.poutine.replay(self.trained_model, trace=guide_trace)
                    logits = replayed_model(images)
                    logits_mc[i] = logits

                avg_logits = logits_mc.mean(dim=0)
                predictions = torch.argmax(avg_logits, dim=1)

                all_labels.extend(labels.cpu().numpy())
                all_predictions.extend(predictions.cpu().numpy())
                all_logits.extend(avg_logits.cpu().numpy())
                all_probs.extend(F.softmax(avg_logits, dim=1).cpu().numpy())

        return all_labels, all_predictions, all_logits, all_probs

    def return_accuracy(self, all_labels, all_predictions):
        cm = confusion_matrix(all_labels, all_predictions)
        return np.trace(cm) / np.sum(cm)

    def compute_softmax_difference(self, before_probs, after_logits, penalty=1.0):
        """
        before_probs: list or array, shape (N, C), all finite probabilities
        after_logits: list or array, shape (N, C), raw logits (may contain ±inf)
        penalty: float, the per‑example penalty to use if logits are nonfinite
        
        Returns the mean over N examples of either
        - max_i |before_probs[n,i] − after_probs[n,i]|,  if after_logits[n] is finite
        - penalty,                                    otherwise
        """
        before = np.asarray(before_probs, dtype=np.float32)
        after_logits = torch.from_numpy(np.asarray(after_logits, dtype=np.float32))
        N, C = after_logits.shape

        # 1) detect which rows of after_logits are all finite
        finite_mask = torch.isfinite(after_logits).all(dim=1).numpy()  # shape (N,)

        # 2) safe‑softmax only on the finite ones
        safe_after_probs = torch.zeros_like(after_logits)
        if finite_mask.any():
            good_logits = after_logits[finite_mask]
            # (you can optionally do the “stable” shift here)
            safe_after_probs[finite_mask] = F.softmax(good_logits, dim=1)
        safe_after_probs = safe_after_probs.numpy()

        # 3) compute per‑example diff, using the penalty where needed
        diffs = np.empty(N, dtype=np.float32)
        for n in range(N):
            if not finite_mask[n]:
                diffs[n] = penalty
            else:
                diffs[n] = np.max(np.abs(before[n] - safe_after_probs[n]))
        return diffs.mean()

    def compute_difference(self, original_val, modified_val):
        return abs(original_val - modified_val)
    
    def run_seu_multivariate(self, location_index, parameter_name, ll_module_index, bit_i, num_samples):
        assert parameter_name in ["loc", "scale"]
        
        pyro.get_param_store().set_state(torch.load(pyro_param_store_path, weights_only=False))

        #param_store_name_initial = f"{param_unique}.{parameter_name}.{layer}.{layer_module}"
        #TODO translate the param_store_name
        # turn things like AutoNormal.locs.conv1.weight into  AutoGuideList.0.loc
        # 0 is conv1.weight, 1 is conv1.bias, etc

        #if f"{layer}.{layer_module}" == "conv1.weight":
        #    layer_module_number = "0"
        #elif f"{layer}.{layer_module}" == "conv1.bias":
        #    layer_module_number = "1"
        #elif f"{layer}.{layer_module}" == "conv2.weight":
        #    layer_module_number = "2"
        #elif f"{layer}.{layer_module}" == "conv2.bias":
        #    layer_module_number = "3"
        #elif f"{layer}.{layer_module}" == "fc1.weight":
        #    layer_module_number = "4"
        #elif f"{layer}.{layer_module}" == "fc1.bias":
        #    layer_module_number = "5"

        #print(f"{layer}.{layer_module}")

        param_store_name = f"AutoGuideList.{ll_module_index}.{parameter_name}"

        with torch.no_grad():
            param = pyro.get_param_store().get_param(param_store_name)
            new_param = param.clone()
            new_param = new_param.view(-1) #flatten new param
            original_val = new_param[location_index].cpu().item()
            seu_val = bitflip_float32(original_val, bit_i)
            abs_diff = self.compute_difference(original_val, seu_val)
            new_param[location_index] = seu_val
            # return new_param to original shape
            new_param = new_param.view(param.shape)
            pyro.get_param_store().__setitem__(param_store_name, new_param)

            print(f"Original value: {original_val}, SEU value: {seu_val}, Abs difference: {abs_diff}")

        guide = AutoGuideList(bayesian_model)

        # 1) conv1.weight
        guide.add(
            AutoLowRankMultivariateNormal(
                poutine.block(bayesian_model, expose=["conv1.weight"]),
                rank=20,
                init_scale=0.05,
                #prefix="AutoGuideList.conv1.weight"
            )
        )

        # 2) conv1.bias
        guide.add(
            AutoLowRankMultivariateNormal(
                poutine.block(bayesian_model, expose=["conv1.bias"]),
                rank=5,
                init_scale=0.05,
                #prefix="AutoGuideList.conv1.bias"
            )
        )

        # 3) conv2.weight
        guide.add(
            AutoLowRankMultivariateNormal(
                poutine.block(bayesian_model, expose=["conv2.weight"]),
                rank=20,
                init_scale=0.05,
                #prefix="AutoGuideList.conv2.weight"
            )
        )

        # 4) conv2.bias
        guide.add(
            AutoLowRankMultivariateNormal(
                poutine.block(bayesian_model, expose=["conv2.bias"]),
                rank=5,
                init_scale=0.05,
                #prefix="AutoGuideList.conv2.bias"
            )
        )

        # 5) fc1.weight
        guide.add(
            AutoLowRankMultivariateNormal(
                poutine.block(bayesian_model, expose=["fc1.weight"]),
                rank=20,
                init_scale=0.05,
                #prefix="AutoGuideList.fc1.weight"
            )
        )

        # 6) fc1.bias
        guide.add(
            AutoLowRankMultivariateNormal(
                poutine.block(bayesian_model, expose=["fc1.bias"]),
                rank=5,
                init_scale=0.05,
                #prefix="AutoGuideList.fc1.bias"
            )
        )

        try:
            after_labels, after_predictions, after_logits, after_probs = self.predict_data_probs(num_samples)
            accuracy_after = self.return_accuracy(after_labels, after_predictions)
            softmax_diff = self.compute_softmax_difference(self.initial_probs, after_probs)
        except:
            print("Error during prediction after SEU.")
            accuracy_after = np.nan
            softmax_diff = np.nan

        print(f"Accuracy after SEU: {accuracy_after}")
        print("===================================")

        return {
            "accuracy_change": accuracy_after - self.initial_accuracy,
            "softmax_difference": softmax_diff,
            "absolute_difference": abs_diff
        }
    
    def run_seu_old(self, location_index, param_unique, parameter_name, layer, layer_module, bit_i, num_samples):
        assert parameter_name in ["locs", "scales", "lows", "widths"], "Parameter name must be 'locs' or 'scales'."
        assert bit_i in range(0, 33), "Bit index must be between 0 and 32."

        param_store_name = f"{param_unique}.{parameter_name}.{layer}.{layer_module}"
        pyro.get_param_store().set_state(torch.load(pyro_param_store_path, weights_only=False))

        with torch.no_grad():
            param = pyro.get_param_store().get_param(param_store_name)
            new_param = param.clone()
            new_param = new_param.view(-1) #flatten new param
            original_val = new_param[location_index].cpu().item()
            seu_val = bitflip_float32(original_val, bit_i)
            abs_diff = self.compute_difference(original_val, seu_val)
            new_param[location_index] = seu_val
            # return new_param to original shape
            new_param = new_param.view(param.shape)
            pyro.get_param_store().__setitem__(param_store_name, new_param)

            print(f"Original value: {original_val}, SEU value: {seu_val}, Abs difference: {abs_diff}")


        if param_unique == "AutoNormal":
            self.guide = AutoNormal(self.trained_model, init_scale=0.05).to(self.device)
        elif param_unique == "AutoLaplace":
            self.guide = AutoLaplace(self.trained_model, init_scale=0.05).to(self.device)
        elif param_unique == "AutoUniform":
            self.guide = AutoUniform(self.trained_model, init_scale=0.05).to(self.device)
        else:
            raise ValueError(f"Unsupported parameter unique: {param_unique}")

        #after_labels, after_predictions, after_logits, after_probs = self.predict_data_probs(num_samples)
        #accuracy_after = self.return_accuracy(after_labels, after_predictions)
        #softmax_diff = self.compute_softmax_difference(self.initial_probs, after_probs)

        try:
            after_labels, after_predictions, after_logits, after_probs = self.predict_data_probs(num_samples)
            accuracy_after = self.return_accuracy(after_labels, after_predictions)
            softmax_diff = self.compute_softmax_difference(self.initial_probs, after_probs)
        except:
            print("Error during prediction after SEU.")
            accuracy_after = np.nan
            softmax_diff = np.nan

        print(f"Accuracy after SEU: {accuracy_after}")
        print("===================================")

        return {
            "accuracy_change": accuracy_after - self.initial_accuracy,
            "softmax_difference": softmax_diff,
            "absolute_difference": abs_diff
        }


    #AutoNormal.locs.conv1.weight
    def run_seu(
        self,
        location_index: int,
        param_unique: str,
        parameter_name: str,
        layer: str,
        layer_module: str,
        bit_i: int,
        num_samples: int,
    ):
        assert parameter_name in ["locs", "scales", "lows", "widths"], \
            "Parameter name must be one of 'locs', 'scales', 'lows', or 'widths'."
        assert 0 <= bit_i < 32, "Bit index must be between 0 and 31."

        # Construct the Pyro ParamStore key for this tensor
        param_store_name = f"{param_unique}.{parameter_name}.{layer}.{layer_module}"

        # insert remarks if any width adjustment (in uniform) or anything is done
        remarks = ""

        # Reload the saved ParamStore so we start from your trained guide
        pyro.clear_param_store()

        pyro.get_param_store().set_state(
            torch.load(pyro_param_store_path, weights_only=False)
        )

        layer_of_interest = layer
        module_of_interest = layer_module

        for name, value in pyro.get_param_store().items():
            if layer_of_interest in name and module_of_interest in name:
                print(f"{name}: {value.cpu().detach().numpy().flatten()[0]}")
        

        with torch.no_grad():
            # 1) Grab the original tensor and flatten it
            param = pyro.get_param_store().get_param(param_store_name)
            flat  = param.clone().view(-1)

            # 2) Extract, flip one bit, and wrap back as a tensor
            orig_val = flat[location_index].cpu().item()
            flipped  = bitflip_float32(orig_val, bit_i)      # Python float

            # if the parameter is 'widths' or 'scales' and the flipped value is negative, we skip
            if parameter_name in ["widths", "scales"] and flipped < 0:
                accuracy_after = np.nan
                softmax_diff = np.nan
                abs_diff = np.nan
                remarks += "SEU width or scale was negative; "
                print(
                    f"Skipping SEU at '{param_store_name}[{location_index}]': "
                    f"original value {orig_val:.3e}, flipped value {flipped:.3e}."
                )
                return {
                    "accuracy_change":    accuracy_after - self.initial_accuracy,
                    "softmax_difference": softmax_diff,
                    "absolute_difference": abs_diff,
                    "remarks": remarks
                }

            # Check for NaN or Inf and clip if necessary
            if np.isnan(flipped) or np.isinf(flipped):
                remarks += "SEU went to NaN or Inf clipping to finite range; "
                print(f"Available clip values range is '{-np.finfo(np.float32).max} {np.finfo(np.float32).max}'")
                #flipped = np.clip(flipped, -np.finfo(np.float32).max, np.finfo(np.float32).max)
                flipped = np.nan_to_num(
                    flipped,
                    nan= np.finfo(np.float32).max if orig_val >= 0 else -np.finfo(np.float32).max,
                    posinf=np.finfo(np.float32).max,
                    neginf=-np.finfo(np.float32).max,
                )
                print(
                    f"Warning: Flipped value at '{param_store_name}[{location_index}]' is NaN or Inf! "
                    f"Original value was {orig_val:.3e}, "
                    f"flipped value is clipped to {flipped:.3e}. Clipping to finite range."
                )

            seu_val  = torch.tensor(
                flipped, dtype=param.dtype, device=param.device
            )
            abs_diff = self.compute_difference(orig_val, flipped)

            # 3) Write the flipped value into the ParamStore
            flat[location_index] = seu_val
            pyro.get_param_store().__setitem__(
                param_store_name, flat.view(param.shape)
            )

            # 4) If this is an AutoUniform guide, enforce
            #    width >= nextafter(low) – low so Uniform(low, low+width) stays valid.
            if param_unique == "AutoUniform":
                
                # (a) get the current lows tensor (after any flip)
                low_name = f"{param_unique}.lows.{layer}.{layer_module}"
                low_param = pyro.get_param_store() \
                                .get_param(low_name) \
                                .view(-1)

                # determine the 'low' value we should use at this index:
                if parameter_name == "lows":
                    low_val = seu_val
                else:  # we just flipped a width, so low stays original
                    low_val = low_param[location_index]

                # check and print if low_val is nan
                if torch.isnan(low_val):
                    print(
                        f"Warning: low value at '{low_name}[{location_index}]' is NaN! "
                        f"Original value was {orig_val:.3e}, flipped value is {flipped:.3e}."
                    )

                # compute the smallest positive increment (ULP) at low_val
                delta = (
                    torch.nextafter(
                        low_val,
                        torch.tensor(float("inf"), dtype=low_val.dtype, device=low_val.device),
                    )
                    - low_val
                )

                # (b) now clamp the corresponding width
                width_name  = f"{param_unique}.widths.{layer}.{layer_module}"
                width_param = pyro.get_param_store().get_param(width_name)
                wflat       = width_param.clone().view(-1)

                # original width at that index (after any SEU if param was 'widths')
                orig_width = wflat[location_index].cpu().item()
                # check whether the orig_width is nan
                if np.isnan(orig_width):
                    print(
                        f"Warning: original width at '{width_name}[{location_index}]' is NaN! "
                        f"Original width is {orig_width:.3e}"
                    )
                # enforce the minimum
                new_width  = torch.max(wflat[location_index], delta)
                wflat[location_index] = new_width

                # check whether the new width is nan
                if torch.isnan(new_width):
                    print(
                        f"Warning: new width at '{width_name}[{location_index}]' is NaN! "
                        f"Original width was {orig_width:.3e}, delta is {delta:.3e}."
                    )

                # write back the clamped widths
                pyro.get_param_store().__setitem__(
                    width_name, wflat.view(width_param.shape)
                )

                if orig_width != new_width:
                    remarks += f"AutoUniform width adjusted; "
                    print(
                        f"Adjusted width at '{width_name}[{location_index}]': "
                        f"{orig_width:.3e} → {new_width:.3e}"
                    )
                else:
                    pass

                if parameter_name == "widths" and seu_val < 0:
                    accuracy_after = np.nan
                    softmax_diff = np.nan
                    abs_diff = np.nan
                    remarks += "SEU width was negative; "
                    # continue or break to skip this?
                    return {
                        "accuracy_change":    accuracy_after - self.initial_accuracy,
                        "softmax_difference": softmax_diff,
                        "absolute_difference": abs_diff,
                        "remarks": remarks
                    }
                    

            # 5) Report the flip
            print(
                f"Parameter '{param_store_name}[{location_index}]': "
                f"{orig_val:.6g} → {flipped:.6g}  "
                f"(abs diff {abs_diff:.3g})"
            )

        # 6) Re‑instantiate your guide so Predictive will pick up the perturbed params
        if param_unique == "AutoNormal":
            self.guide = AutoNormal(self.trained_model, init_scale=0.05).to(self.device)
        elif param_unique == "AutoLaplace":
            self.guide = AutoLaplace(self.trained_model, init_scale=0.05).to(self.device)
        elif param_unique == "AutoUniform":
            self.guide = AutoUniform(self.trained_model, init_scale=0.05).to(self.device)
        else:
            raise ValueError(f"Unsupported guide type: {param_unique}")

        # 7) Re‑run inference & evaluation
        after_labels, after_preds, after_logits, after_probs = self.predict_data_probs(num_samples)
        accuracy_after = self.return_accuracy(after_labels, after_preds)
        softmax_diff  = self.compute_softmax_difference(self.initial_probs, after_probs)

        print(f"Accuracy after SEU: {accuracy_after:.3%}")
        print("===================================")
        
        for name, value in pyro.get_param_store().items():
            if layer_of_interest in name and module_of_interest in name:
                print(f"{name}: {value.cpu().detach().numpy().flatten()[0]}")

        return {
            "accuracy_change":    accuracy_after - self.initial_accuracy,
            "softmax_difference": softmax_diff,
            "absolute_difference": abs_diff,
            "remarks": remarks
        }


    def run_seu_autodiagonal_normal_multi(self, location_indices, bit_i, parameter_name="loc",
                                          attack_ratio=1.0, num_samples=10, seed=None):
        assert parameter_name in ["loc", "scale"], "Parameter name must be 'loc' or 'scale'."
        assert bit_i in range(0, 33), "Bit index must be between 0 and 32."
        assert 0.0 <= attack_ratio <= 1.0, "Attack ratio must be between 0.0 and 1.0."

        if isinstance(location_indices, int):
            location_indices = [location_indices]

        if seed is not None:
            np.random.seed(seed)
            torch.manual_seed(seed)

        num_attacks = max(1, int(len(location_indices) * attack_ratio))
        attack_locations = np.random.choice(location_indices, size=num_attacks, replace=False)
        param_store_name = f"AutoDiagonalNormal.{parameter_name}"
        pyro.get_param_store().set_state(torch.load(pyro_param_store_path, weights_only=False))

        abs_differences = []

        with torch.no_grad():
            param = pyro.get_param_store().get_param(param_store_name)
            new_param = param.clone()

            #print(f"Attacking {num_attacks} out of {len(location_indices)} locations:")

            for location_index in attack_locations:
                original_val = new_param[location_index].cpu().item()
                seu_val = bitflip_float32(original_val, bit_i)
                abs_diff = self.compute_difference(original_val, seu_val)
                abs_differences.append(abs_diff)
                new_param[location_index] = seu_val
                print(f"  Location {location_index}: {original_val} -> {seu_val}, Log diff: {abs_diff}")

            pyro.get_param_store().__setitem__(param_store_name, new_param)

        self.guide = AutoDiagonalNormal(self.trained_model).to(self.device)

        try:
            after_labels, after_predictions, after_logits, after_probs = self.predict_data_probs(num_samples)
            accuracy_after = self.return_accuracy(after_labels, after_predictions)
            softmax_diff = self.compute_softmax_difference(self.initial_probs, after_probs)
            mean_abs_diff = float(np.mean(abs_differences))
        except:
            accuracy_after = np.nan
            softmax_diff = np.nan
            mean_abs_diff = np.nan

        #print(f"Accuracy after SEU: {accuracy_after}")
        #print("===================================")

        return {
            "accuracy_change": accuracy_after - self.initial_accuracy,
            "softmax_difference": softmax_diff,
            "mean_abs_difference": mean_abs_diff
        }

def load_model_config(timestamp):
    config_path = os.path.join(search_dir, config_files[timestamp])

    with open(config_path, 'r') as f:
        model_config = json.load(f)

    return model_config

In [23]:
prior_interest = "Uniform_prior"

train_loader, test_loader = load_data(batch_size=16)
device = torch.device("cuda")
num_classes = 2


search_dir = os.path.join("results_shipsnet_new_uniform","results_shipsnet_v02_01")
save_dir = os.path.join("results_shipsnet_new_uniform","results_shipsnet_v02_01_test")
#list all .json files in the directory
all_files = [f for f in os.listdir(search_dir)]
json_files = [f for f in os.listdir(search_dir) if f.endswith('.json')]

# excluding the format, get the last 16 characters of each filename
timestamps = [f[:-5][-16:] for f in json_files]
print("Timestamps found count:", len(timestamps))

# remove some timestamps that are not needed
# those are the ones that are not in the shipsnet_seu_result directory, without the .csv extension
excluded_timestamps = [f[:-4][-16:] for f in [f for f in os.listdir(save_dir) if f.endswith('.csv')]]

timestamps = [ts for ts in timestamps if ts not in excluded_timestamps]
#timestamps = timestamps[:1]
print("After excluding, timestamps count:", len(timestamps))

# for each timestamp, look for every other files in the directory that contains the timestamp

config_files = {}
guide_files = {}
model_files = {}
param_files = {}

for timestamp in timestamps:
    config_files[timestamp] = [f for f in all_files if timestamp in f and f.endswith('.json')][0]
    guide_files[timestamp] = [f for f in all_files if timestamp in f and f.startswith('guide')][0]
    model_files[timestamp] = [f for f in all_files if timestamp in f and f.startswith('model')][0]
    param_files[timestamp] = [f for f in all_files if timestamp in f and f.startswith('param')][0]

# create a list that maps timestamps and the output of load_model_config['prior']
prior_list = []
for ts in timestamps:
    model_config = load_model_config(ts)
    prior_list.append((ts, model_config['prior']))

# remove timestamps which prior is not stated in args.prior
# translate what args.prior specified
prior_map = {
    'Gaussian_prior': 'gaussian',
    'Laplace_prior': 'laplace',
    'Uniform_prior': 'uniform'
}



timestamps = [ts for ts, prior in prior_list if prior == prior_map[prior_interest]]
print(f"After filtering by prior '{prior_interest}', timestamps count: {len(timestamps)}")


experiment_iteration = 0

Timestamps found count: 63
After excluding, timestamps count: 63
After filtering by prior 'Uniform_prior', timestamps count: 21


In [24]:
for ts_idx, timestamp in enumerate(timestamps):
    model_config = load_model_config(timestamp)

    # Check if the model config matches the expected structure
    if model_config['activation'] == 'relu' and model_config['prior'] == 'uniform' and model_config['prior_params']['b'] == 1.0:
        print(f"Timestamp: {timestamp}, Activation: {model_config['activation']}, Prior: {model_config['prior']}")
        print(f"Prior Params: mu={model_config['prior_params']['mu']}, b={model_config['prior_params']['b']}")
        print("Model Config:", model_config)
        print("-" * 50)
        print("ts_idx:", ts_idx)
        ts_idx_set = ts_idx

Timestamp: _20250806_013752, Activation: relu, Prior: uniform
Prior Params: mu=0.0, b=1.0
Model Config: {'activation': 'relu', 'prior': 'uniform', 'num_epochs': 100, 'best_accuracy_at_epoch': 50, 'best_accuracy': 0.8684375, 'batch_size': 16, 'train_size': 3200, 'prior_params': {'mu': 0.0, 'b': 1.0}}
--------------------------------------------------
ts_idx: 10


In [25]:
pyro.clear_param_store()


bayesian_model, pyro_param_store_path = load_model(timestamps[ts_idx_set])

newinj = NewInjector(trained_model=bayesian_model, device=device, test_loader=test_loader, num_samples=10)

Loading model with config_path: results_shipsnet_new_uniform\results_shipsnet_v02_01\config_relu_uniform_20250806_013752.json
[INFO] Using prior: uniform (mu=0.0, b=1.0)


Evaluating:   0%|          | 0/50 [00:00<?, ?it/s]

'conv1.bias' sampled value (batch 16):
tensor([-0.3367, -0.6216, -0.5816, -0.3765,  0.1322, -0.2538, -0.4603, -0.2176,
        -0.4961, -0.1663, -0.0405, -0.0475, -0.2221, -0.1882, -0.4170, -0.5304,
        -0.5029, -0.1152, -0.4252, -0.2377, -0.0162, -0.2567, -0.3210, -0.2907,
        -0.3704, -0.2794,  0.0353, -0.4976, -0.2471, -0.3852, -0.3346,  0.0143],
       device='cuda:0')


Evaluating:   4%|▍         | 2/50 [00:25<08:34, 10.72s/it]

'conv1.bias' sampled value (batch 16):
tensor([-0.1551, -0.4430, -0.4328, -0.4148,  0.1397, -0.1762, -0.3428, -0.5547,
        -0.3078, -0.1569, -0.3480, -0.1969, -0.5415, -0.3916, -0.3123, -0.2759,
        -0.1580, -0.3016, -0.3629,  0.0133, -0.3443, -0.0447, -0.0768, -0.3367,
        -0.2258, -0.3297, -0.2762, -0.2116, -0.1958, -0.3123, -0.0288, -0.0922],
       device='cuda:0')
'conv1.bias' sampled value (batch 16):
tensor([-0.2418, -0.4307, -0.5101, -0.1049,  0.1470, -0.1053, -0.4091, -0.1798,
        -0.2927, -0.1834,  0.0180, -0.1452, -0.4970, -0.2559, -0.1591, -0.5711,
        -0.2504, -0.1538,  0.0137, -0.3565, -0.3911, -0.1971, -0.3961, -0.0799,
        -0.4402, -0.2909,  0.2299, -0.1705, -0.3259, -0.6382, -0.3242, -0.1736],
       device='cuda:0')


...

'conv1.bias' sampled value (batch 16):
tensor([-0.3210, -0.7002, -0.5073, -0.3509,  0.0293, -0.4305, -0.2289, -0.2725,
        -0.4143, -0.4319, -0.0402, -0.2962, -0.2140, -0.0752, -0.2051, -0.3097,
        -0.2449, -0.4049, -0.0518, -0.2585, -0.2550, -0.4437, -0.1005, -0.1657,
        -0.5193, -0.1429,  0.1653, -0.2830, -0.0026, -0.6404, -0.2562,  0.2334],
       device='cuda:0')
'conv1.bias' sampled value (batch 16):
tensor([-0.3868, -0.4314, -0.5746, -0.5157, -0.0036, -0.2573, -0.5621, -0.4458,
        -0.4731, -0.2955, -0.2669, -0.2069, -0.4457, -0.2262, -0.0424, -0.3337,
        -0.3278,  0.0092, -0.4456, -0.4630, -0.0144, -0.2200, -0.1932, -0.3579,
        -0.1657,  0.0231, -0.0738, -0.3018, -0.1935, -0.7071, -0.3088, -0.2868],
       device='cuda:0')
'conv1.bias' sampled value (batch 16):
tensor([-0.3443, -0.3705, -0.5246, -0.5127,  0.1574, -0.2814, -0.1733, -0.1256,
        -0.2043, -0.3385,  0.0356, -0.3431, -0.4587, -0.3410, -0.0959, -0.4149,
        -0.4275, -0.4094, -0.0150

Evaluating:  96%|█████████▌| 48/50 [00:29<00:00, 14.87it/s]

'conv1.bias' sampled value (batch 16):
tensor([-0.2854, -0.7330, -0.6120, -0.4348, -0.2428, -0.2216, -0.5440, -0.2442,
        -0.2720, -0.1775, -0.1203, -0.4722, -0.3198, -0.4238, -0.2389, -0.1784,
        -0.0927, -0.1504, -0.1011, -0.0896, -0.0737, -0.2722, -0.0071, -0.2532,
        -0.2721, -0.2337, -0.2904, -0.1836, -0.1855, -0.3548, -0.0305, -0.2502],
       device='cuda:0')
'conv1.bias' sampled value (batch 16):
tensor([-0.1597, -0.6549, -0.3210, -0.5616,  0.1818, -0.3641, -0.3715, -0.2304,
        -0.5048, -0.4500, -0.1583, -0.0866, -0.5970, -0.5329, -0.3042, -0.3110,
        -0.4161, -0.1596, -0.3014, -0.4062, -0.4280, -0.1636, -0.3355, -0.1392,
        -0.5774, -0.2667, -0.1986, -0.1348, -0.1033, -0.4834, -0.0338, -0.1143],
       device='cuda:0')
'conv1.bias' sampled value (batch 16):
tensor([-0.3158, -0.6545, -0.5172, -0.2939,  0.2039, -0.0079, -0.1296, -0.2598,
        -0.4742, -0.3602,  0.0759, -0.5139, -0.2306, -0.2336, -0.0813, -0.5321,
        -0.4558, -0.1155, -0.3345

Evaluating: 100%|██████████| 50/50 [00:30<00:00,  1.66it/s]


'conv1.bias' sampled value (batch 16):
tensor([-0.1538, -0.5653, -0.4911, -0.4292,  0.2152, -0.2970, -0.1286, -0.1793,
        -0.3768, -0.5142, -0.2605, -0.0667, -0.3591, -0.2386, -0.3536, -0.2645,
        -0.4075, -0.2926, -0.0136, -0.1293, -0.3806, -0.0390, -0.4324, -0.2461,
        -0.3446, -0.0622,  0.1075, -0.1338, -0.3317, -0.3556, -0.4727, -0.1193],
       device='cuda:0')
'conv1.bias' sampled value (batch 16):
tensor([-0.0824, -0.3237, -0.6912, -0.4198,  0.1686, -0.4226, -0.3008, -0.1441,
        -0.3353, -0.4030, -0.0453, -0.1884, -0.4572, -0.1112, -0.2133, -0.2107,
        -0.3762, -0.4133, -0.1654, -0.0196, -0.1351, -0.3768, -0.3301, -0.4026,
        -0.3969, -0.4158, -0.2305, -0.3113, -0.3110, -0.5214, -0.0774,  0.0417],
       device='cuda:0')
Initial accuracy: 0.9025


In [26]:
target_index = 0 # attack the first neuron
param_unique_target = "AutoUniform" 
parameter_name = "lows"
layer_list_iter ="conv1"
module_iter = "bias"
bit_iter = 1

result = newinj.run_seu(target_index, param_unique_target, parameter_name, layer_list_iter, module_iter, bit_iter, num_samples=10)

AutoUniform.lows.conv1.bias: -0.5120815634727478
AutoUniform.widths.conv1.bias: 0.4894312620162964
Adjusted width at 'AutoUniform.widths.conv1.bias[0]': 4.894e-01 → 2.028e+31
Parameter 'AutoUniform.lows.conv1.bias[0]': -0.512082 → -1.74252e+38  (abs diff 1.74e+38)


Evaluating:   2%|▏         | 1/50 [00:00<00:07,  6.69it/s]

'conv1.bias' sampled value (batch 16):
tensor([-1.7425e+38, -3.6787e-01, -6.5465e-01, -3.3160e-01,  3.0419e-01,
        -1.1283e-01, -5.2393e-01, -3.4333e-01, -2.3855e-01, -2.6094e-01,
        -5.4032e-02, -5.1181e-01, -4.3184e-01, -2.5458e-01, -4.0569e-01,
        -4.5194e-01, -1.2785e-01, -4.3397e-01, -3.2551e-02, -1.6660e-01,
        -1.6334e-01,  9.5741e-03, -3.9393e-02, -3.2865e-01, -3.5322e-01,
        -5.2720e-02,  2.3907e-01, -2.4332e-01, -1.4014e-01, -4.9874e-01,
        -2.4132e-01,  1.0925e-01], device='cuda:0')
'conv1.bias' sampled value (batch 16):
tensor([-1.7425e+38, -3.3892e-01, -5.2329e-01, -4.0863e-01,  1.6343e-01,
        -1.4368e-01, -2.7714e-01, -1.2594e-01, -3.1646e-01, -4.9988e-01,
        -2.4839e-01, -1.0119e-01, -3.8758e-01, -3.0559e-01, -3.2484e-01,
        -3.2446e-01, -3.7031e-01, -3.4214e-01, -2.3764e-01, -3.3848e-01,
        -4.3379e-01, -2.1738e-01, -7.6161e-02, -3.5324e-01, -3.7530e-01,
        -7.8938e-02,  1.1545e-01, -4.5178e-01, -1.3381e-01, -4.8302

Evaluating:   6%|▌         | 3/50 [00:00<00:06,  6.81it/s]

'conv1.bias' sampled value (batch 16):
tensor([-1.7425e+38, -4.9543e-01, -6.5687e-01, -2.4512e-01, -4.5398e-02,
        -3.9070e-01, -1.4723e-01, -3.6018e-01, -3.4221e-01, -5.8877e-01,
        -3.1781e-01, -4.7761e-01, -1.8593e-01, -4.6659e-01, -4.6063e-01,
        -1.6946e-01, -2.1054e-01, -2.6880e-01, -3.0900e-01, -4.6718e-01,
        -3.0278e-01, -1.6118e-01, -4.1768e-01, -1.6100e-01, -4.2593e-01,
        -4.2669e-01, -2.0084e-01, -3.3260e-01, -1.0593e-01, -5.3064e-01,
        -2.6990e-01,  7.4822e-02], device='cuda:0')
'conv1.bias' sampled value (batch 16):
tensor([-1.7425e+38, -5.6084e-01, -5.5851e-01, -2.8293e-01,  2.6103e-01,
        -7.5818e-02, -3.5807e-01, -4.1168e-01, -3.1840e-01, -2.5234e-01,
        -2.2941e-01, -1.2226e-01, -5.8622e-01, -1.2095e-01, -1.9323e-01,
        -5.7627e-01, -4.7033e-01, -4.1196e-01, -3.6545e-01, -3.6696e-01,
        -2.9788e-01, -4.3779e-01, -4.8465e-01, -3.7824e-01, -5.2665e-01,
        -3.3231e-01, -2.0563e-01, -1.0805e-01,  4.7402e-02, -4.9801

...

'conv1.bias' sampled value (batch 16):
tensor([-1.7425e+38, -6.2384e-01, -4.1884e-01, -5.6062e-01, -1.4276e-01,
        -1.1295e-01, -1.6223e-01, -5.8716e-01, -5.4521e-01, -2.7598e-01,
        -4.1030e-01, -4.3137e-01, -1.9950e-01, -1.7860e-01, -3.7325e-01,
        -4.1439e-01, -1.2386e-01, -2.1699e-01, -4.4648e-01, -1.3429e-02,
        -3.1107e-01, -2.0970e-01, -3.6817e-01, -3.1364e-02, -5.9299e-01,
        -2.7758e-01, -7.7406e-02, -3.7384e-01, -1.0426e-01, -6.4713e-01,
        -1.4376e-01, -1.0368e-01], device='cuda:0')
'conv1.bias' sampled value (batch 16):
tensor([-1.7425e+38, -5.8583e-01, -5.8737e-01, -5.3808e-01,  1.8250e-01,
        -4.8575e-02, -2.6020e-01, -4.9869e-01, -3.5962e-01, -2.9112e-01,
         3.3122e-02, -2.2341e-01, -2.7483e-01, -1.9484e-01, -1.3047e-01,
        -3.5091e-01, -3.0714e-01, -2.9985e-01, -4.1384e-01, -4.0264e-01,
        -3.0405e-01, -2.1443e-01, -4.0534e-01,  6.2918e-02, -4.0719e-01,
        -2.6658e-01, -2.1783e-01, -2.7113e-01, -2.6256e-01, -5.4334

Evaluating:  98%|█████████▊| 49/50 [00:03<00:00, 15.12it/s]

'conv1.bias' sampled value (batch 16):
tensor([-1.7425e+38, -6.4109e-01, -4.0838e-01, -4.3829e-01, -1.8852e-01,
        -2.4755e-01, -1.3186e-01, -2.5116e-01, -3.4707e-01, -5.4755e-01,
        -4.0045e-01, -9.1855e-02, -4.9578e-01, -3.8636e-01, -1.5514e-01,
        -1.7631e-01, -2.9779e-01, -5.6080e-03, -1.1428e-01,  4.0185e-04,
        -3.1358e-01,  1.6888e-02, -2.7612e-01, -2.7030e-01, -4.0268e-01,
        -3.4466e-02, -5.6660e-02, -3.6330e-01, -4.1327e-02, -5.5282e-01,
        -3.9204e-01, -3.1757e-01], device='cuda:0')
'conv1.bias' sampled value (batch 16):
tensor([-1.7425e+38, -5.2774e-01, -3.0630e-01, -2.9417e-01, -4.9961e-02,
        -2.0426e-01, -5.5267e-01, -1.7531e-01, -4.1799e-01, -2.1755e-01,
        -2.8322e-01, -3.9692e-01, -5.5017e-01, -7.6945e-02, -1.4855e-01,
        -3.2790e-01, -1.0539e-01, -1.2620e-01, -2.4533e-01, -5.6390e-02,
        -2.8428e-01, -4.0281e-01, -3.5159e-01, -8.7280e-02, -5.1134e-01,
        -2.0834e-01,  1.5732e-01, -3.6366e-01, -8.8326e-03, -4.7073

Evaluating: 100%|██████████| 50/50 [00:03<00:00, 12.70it/s]

'conv1.bias' sampled value (batch 16):
tensor([-1.7425e+38, -4.4091e-01, -5.6025e-01, -3.1608e-01, -1.3201e-01,
        -1.3760e-01, -1.7882e-01, -2.4239e-01, -1.9873e-01, -4.0169e-01,
        -3.6590e-03, -2.8507e-01, -3.3761e-01, -1.1541e-01, -5.2163e-01,
        -1.6173e-01, -5.4588e-01, -2.4669e-01, -2.3039e-01,  1.1206e-02,
        -5.4460e-02, -3.9793e-01, -5.8439e-02,  4.8581e-02, -3.8032e-01,
        -1.1661e-01,  2.2639e-01, -7.2421e-02, -2.5881e-01, -5.9601e-01,
        -1.5457e-01, -1.7125e-01], device='cuda:0')
Accuracy after SEU: 91.000%
AutoUniform.lows.conv1.bias: -1.742523264750814e+38
AutoUniform.widths.conv1.bias: 2.028240960365167e+31


In [ ]:
import pyro
import torch

In [ ]:
layer_of_interest = "conv1"
module_of_interest = "weight"

for name, value in pyro.get_param_store().items():
    if layer_of_interest in name and module_of_interest in name:
        print(f"{name}: {value.cpu().detach().numpy()[0][0][0][0]}")

In [ ]:
for name, value in pyro.get_param_store().items():
    print(f"{name}:")